# Репрезентативное превью IRT-датасета

Проверка пайплайна **yaml → `IRTDataset` → фичи (TSR/Fourier) → ROI-кроп → маска**.

Что внутри:
1. состав датасета (видео, размеры, маски, ROI в meta);
2. один сэмпл целиком: ключи, метаданные, все каналы;
3. много кропов с оверлеем маски и meta в заголовке;
4. deep-dive: **кадры исходного `.mat`**, полная маска, `object_crop` / ROI, рядом фичи и финальный кроп;
5. статистики (доля дефекта, пустые маски, диапазоны каналов).

По умолчанию: `segmentation/U-Net/dataset_tsr.yaml`, `train=False` (без аугментаций).


In [ ]:
from __future__ import annotations

import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import torch

ROOT = Path("..").resolve()
UNET = ROOT / "segmentation" / "U-Net"
SEG = ROOT / "segmentation"
for p in (ROOT, UNET, SEG):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from irt_cfg import load_cfg
from irt_data import IRTDataset

# --- knobs ---
YAML = UNET / "dataset_tsr.yaml"  # dataset_fourier.yaml / dataset.yaml / dataset_samples.yaml
TRAIN = False                     # False = без augs
N_GRID = 36                       # кропов на общей сетке
N_DEEP = 8                        # роликов с исходными кадрами
SEED = 0
FRAME_POSITIONS = (0.05, 0.35, 0.65, 0.95)

cfg = load_cfg(YAML, train=TRAIN)
ds = IRTDataset(cfg)
s0 = ds[0]
img0 = s0["image"]
print(f"yaml     : {YAML.relative_to(ROOT)}")
print(f"mode     : {cfg.mode} | train={cfg.train}")
print(f"features : {cfg.features.extractors} | poly={cfg.features.poly_degree} | ppt_bins={cfg.features.ppt_bins}")
print(f"crop     : {cfg.crop.strategy} size={cfg.crop.size} | samples_per_video={cfg.samples_per_video}")
print(f"masks src: {[s.masks for s in cfg.sources]}")
print(f"len(ds)  : {len(ds)}  (= {len(ds.video_ids)} videos × samples_per_video)")
print(f"sample0  : image={tuple(img0.shape)} mask={tuple(s0['mask'].shape)} id={s0['video_id']!r}")
print(f"keys     : {sorted(s0.keys())}")


## 1. Инвентаризация: видео, формы, маски, ROI


In [ ]:
def series_prefix(vid: str) -> str:
    if vid.startswith("sample"):
        return "sample"
    return vid.split("_", 1)[0] if "_" in vid else vid

rows = []
missing_mask, missing_meta = [], []
for vid in ds.video_ids:
    T, H, W = ds.backend.shape(vid)
    mask, has = ds._load_mask(vid, H, W)
    meta = ds._meta(vid)
    rois = list(meta.rois) if meta is not None and meta.rois else []
    if not has:
        missing_mask.append(vid)
    if meta is None:
        missing_meta.append(vid)
    rows.append({
        "video_id": vid,
        "prefix": series_prefix(vid),
        "T": T, "H": H, "W": W,
        "has_mask": bool(has),
        "mask_px": int((mask > 0).sum()) if has else 0,
        "mask_frac": float((mask > 0).mean()) if has else 0.0,
        "n_rois": len(rois),
        "cool_start": getattr(meta, "cool_start", None) if meta else None,
    })

df = pd.DataFrame(rows)
print("по префиксу:", dict(Counter(df["prefix"])))
print(f"без маски ({len(missing_mask)}):", missing_mask[:10] or "—")
print(f"без files_meta ({len(missing_meta)}):", missing_meta[:10] or "—")

display(df.groupby("prefix").agg(
    n=("video_id", "count"),
    T_med=("T", "median"),
    H=("H", "first"),
    W=("W", "first"),
    mask_frac_mean=("mask_frac", "mean"),
    mask_frac_min=("mask_frac", "min"),
    n_rois_mean=("n_rois", "mean"),
).round(4))
display(df.sort_values(["prefix", "video_id"]).reset_index(drop=True))


## 2. Один сэмпл: метаданные + все каналы фич + маска


In [ ]:
def tensor_info(t: torch.Tensor) -> str:
    a = t.detach().float().cpu().numpy()
    return f"shape={tuple(t.shape)} min={a.min():.4g} max={a.max():.4g} mean={a.mean():.4g}"

s = ds[0]
vid = s["video_id"]
crop = [int(x) for x in s["crop"].tolist()]   # y0, x0, y1, x1
obj = [int(x) for x in s["object_crop"].tolist()]
print("video_id      :", vid)
print("has_mask      :", bool(s["has_mask"].item() if hasattr(s["has_mask"], "item") else s["has_mask"]))
print("crop y0x0y1x1 :", crop)
print("object_crop   :", obj)
print("frame_indices :", s["frame_indices"].tolist())
print("image         :", tensor_info(s["image"]))
print("mask          :", tensor_info(s["mask"]))
meta = ds._meta(vid)
if meta is not None:
    print("meta.rois     :", [(r.x, r.y, r.w, r.h) for r in (meta.rois or [])])
    print("meta.cool_start:", meta.cool_start)
    print("defect/swap boxes:", len(meta.swap_boxes()) if hasattr(meta, "swap_boxes") else "n/a")

img = s["image"].numpy()
msk = s["mask"].numpy()
if msk.ndim == 3:
    msk = msk[0]
C = img.shape[0]
ncols = min(C + 1, 7)
nrows = int(np.ceil((C + 1) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(2.5 * ncols, 2.5 * nrows))
axes = np.atleast_1d(axes).ravel()
for c in range(C):
    axes[c].imshow(img[c], cmap="inferno")
    if msk.max() > 0:
        axes[c].contour(msk, levels=[0.5], colors="cyan", linewidths=0.9)
    axes[c].set_title(f"ch{c}", fontsize=9)
    axes[c].axis("off")
axes[C].imshow((msk > 0).astype(float), cmap="gray", vmin=0, vmax=1)
axes[C].set_title(f"mask ({(msk > 0).sum()}px)", fontsize=9)
axes[C].axis("off")
for ax in axes[C + 1:]:
    ax.axis("off")
fig.suptitle(f"{vid} | crop={crop}", fontsize=11)
plt.tight_layout()
plt.show()


## 3. Много кропов (ch0 + контур маски + meta)


In [ ]:
rng = np.random.default_rng(SEED)
by_vid: dict[str, list[int]] = {}
for i, v in enumerate(ds._index):
    by_vid.setdefault(v, []).append(i)
vids = list(by_vid)
rng.shuffle(vids)
pick = []
for v in vids:
    pick.append(int(rng.choice(by_vid[v])))
    if len(pick) >= N_GRID:
        break

ncols = 6
nrows = int(np.ceil(len(pick) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(2.7 * ncols, 2.9 * nrows))
axes = np.atleast_2d(axes)

for ax, i in zip(axes.ravel(), pick):
    s = ds[i]
    img = s["image"].numpy()
    m = s["mask"].numpy()
    if m.ndim == 3:
        m = m[0]
    mbin = (m > 0).astype(float)
    ax.imshow(img[0], cmap="inferno")
    if mbin.max() > 0:
        ax.contour(mbin, levels=[0.5], colors="lime", linewidths=1.0)
    y0, x0, y1, x1 = [int(x) for x in s["crop"].tolist()]
    ax.set_title(
        f"{s['video_id']}\n#{i} mask={mbin.sum():.0f}px\ncrop=({y0},{x0})-({y1},{x1})",
        fontsize=7,
    )
    ax.axis("off")

for ax in axes.ravel()[len(pick):]:
    ax.axis("off")
fig.suptitle(f"grid n={len(pick)} | {YAML.name} | train={TRAIN}", fontsize=12)
plt.tight_layout()
plt.show()


## 4. Deep-dive: исходное видео → object_crop / ROI → фичи → кроп

Слева направо:
1. кадры из `.mat` (начало → конец), контур **полной** маски, красный = `object_crop`, жёлтый = ROI из `files_meta`;
2. все каналы фич финального кропа + маска кропа.

`object_crop` / `crop` в сэмпле: `(y0, x0, y1, x1)` (y1/x1 exclusive).


In [ ]:
def rect_yx(y0, x0, y1, x1):
    return x0, y0, max(0, x1 - x0), max(0, y1 - y0)

def draw_box(ax, y0, x0, y1, x1, color, label=None, lw=1.6):
    x, y, w, h = rect_yx(y0, x0, y1, x1)
    ax.add_patch(mpatches.Rectangle((x, y), w, h, fill=False, edgecolor=color, lw=lw, label=label))

def show_deep(video_id: str):
    # первый индекс этого видео в датасете
    sample_index = next(i for i, v in enumerate(ds._index) if v == video_id)
    s = ds[sample_index]
    T, H, W = ds.backend.shape(video_id)
    idxs = [min(T - 1, max(0, int(p * (T - 1)))) for p in FRAME_POSITIONS]
    frames = ds.backend.read_frames(video_id, idxs)
    mask_full, has = ds._load_mask(video_id, H, W)
    mbin = (mask_full > 0).astype(float) if has else np.zeros((H, W), np.float32)
    meta = ds._meta(video_id)
    oy0, ox0, oy1, ox1 = [int(x) for x in s["object_crop"].tolist()]
    cy0, cx0, cy1, cx1 = [int(x) for x in s["crop"].tolist()]

    img = s["image"].numpy()
    mcrop = s["mask"].numpy()
    if mcrop.ndim == 3:
        mcrop = mcrop[0]
    mcrop_b = (mcrop > 0).astype(float)
    C = img.shape[0]

    n_frames = len(idxs)
    ncols = max(n_frames, C + 1)
    fig = plt.figure(figsize=(2.3 * ncols, 5.2))
    # row 0: raw frames
    for j, (fi, fr) in enumerate(zip(idxs, frames)):
        ax = fig.add_subplot(2, ncols, j + 1)
        ax.imshow(fr, cmap="inferno")
        if mbin.max() > 0:
            ax.contour(mbin, levels=[0.5], colors="cyan", linewidths=0.8)
        draw_box(ax, oy0, ox0, oy1, ox1, "red", label="object_crop" if j == 0 else None)
        if meta is not None and meta.rois:
            for k, r in enumerate(meta.rois):
                ax.add_patch(mpatches.Rectangle(
                    (r.x, r.y), r.w, r.h, fill=False, edgecolor="yellow", lw=1.2,
                    label="files_meta.roi" if (j == 0 and k == 0) else None,
                ))
        ax.set_title(f"t={fi}/{T-1}", fontsize=9)
        ax.axis("off")
        if j == 0:
            ax.legend(loc="upper right", fontsize=7, framealpha=0.7)
    for j in range(n_frames, ncols):
        fig.add_subplot(2, ncols, j + 1).axis("off")

    # row 1: feature channels + crop mask
    for c in range(C):
        ax = fig.add_subplot(2, ncols, ncols + c + 1)
        ax.imshow(img[c], cmap="inferno")
        if mcrop_b.max() > 0:
            ax.contour(mcrop_b, levels=[0.5], colors="lime", linewidths=0.9)
        ax.set_title(f"feat ch{c}", fontsize=9)
        ax.axis("off")
    ax = fig.add_subplot(2, ncols, ncols + C + 1)
    ax.imshow(mcrop_b, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"crop mask\n{mcrop_b.sum():.0f}px", fontsize=9)
    ax.axis("off")
    for j in range(C + 1, ncols):
        fig.add_subplot(2, ncols, ncols + j + 1).axis("off")

    fig.suptitle(
        f"{video_id} | raw {H}×{W} T={T} | object_crop=({oy0},{ox0})-({oy1},{ox1}) | "
        f"feat_crop=({cy0},{cx0})-({cy1},{cx1}) | has_mask={has}",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()

rng_d = np.random.default_rng(SEED + 7)
deep_vids = list(ds.video_ids)
rng_d.shuffle(deep_vids)
deep_vids = deep_vids[:N_DEEP]
# гарантируем разнообразие префиксов
pref_seen = set()
ordered = []
for v in list(ds.video_ids):
    p = series_prefix(v)
    if p not in pref_seen:
        ordered.append(v)
        pref_seen.add(p)
for v in deep_vids:
    if v not in ordered:
        ordered.append(v)
deep_vids = ordered[:N_DEEP]
print("deep-dive:", deep_vids)

for vid in deep_vids:
    show_deep(vid)


## 5. Статистики по сэмплам (маска / каналы)


In [ ]:
# сэмплируем до 200 индексов, чтобы не тормозить
rng_s = np.random.default_rng(SEED + 3)
n_stats = min(200, len(ds))
stat_idx = rng_s.choice(len(ds), size=n_stats, replace=False)

mask_fracs = []
empty = 0
ch_mins, ch_maxs, ch_means = [], [], []
ids = []
for i in stat_idx:
    s = ds[int(i)]
    m = s["mask"].numpy()
    if m.ndim == 3:
        m = m[0]
    frac = float((m > 0).mean())
    mask_fracs.append(frac)
    if frac == 0:
        empty += 1
    img = s["image"].numpy()
    ch_mins.append(img.min(axis=(1, 2)))
    ch_maxs.append(img.max(axis=(1, 2)))
    ch_means.append(img.mean(axis=(1, 2)))
    ids.append(s["video_id"])

mask_fracs = np.asarray(mask_fracs)
ch_mins = np.stack(ch_mins)
ch_maxs = np.stack(ch_maxs)
ch_means = np.stack(ch_means)
C = ch_means.shape[1]

print(f"stats on {n_stats} samples | empty masks: {empty}/{n_stats} ({100*empty/n_stats:.1f}%)")
print(f"mask_frac: mean={mask_fracs.mean():.4f} median={np.median(mask_fracs):.4f} "
      f"p10={np.percentile(mask_fracs,10):.4f} p90={np.percentile(mask_fracs,90):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(mask_fracs, bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("доля дефекта в кропе")
axes[0].set_xlabel("mask fraction")
axes[0].grid(alpha=0.3)

x = np.arange(C)
axes[1].plot(x, ch_means.mean(0), "o-", label="mean")
axes[1].fill_between(x, ch_mins.mean(0), ch_maxs.mean(0), alpha=0.2, label="avg min..max")
axes[1].set_title("каналы фич (усреднено по сэмплам)")
axes[1].set_xlabel("channel")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

# по сериям
pdf = pd.DataFrame({"video_id": ids, "mask_frac": mask_fracs})
pdf["prefix"] = pdf["video_id"].map(series_prefix)
display(pdf.groupby("prefix")["mask_frac"].agg(["count", "mean", "median", "min", "max"]).round(4))


## 6. (Опционально) train=True — один кроп с аугментациями

Перезагружает датасет с `train=True` и показывает несколько повторных взятий **одного и того же индекса** — должны отличаться кроп/ауги.


In [ ]:
cfg_tr = load_cfg(YAML, train=True)
ds_tr = IRTDataset(cfg_tr)
idx = 0
vid = ds_tr._index[idx]
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax in axes.ravel():
    s = ds_tr[idx]
    img = s["image"].numpy()
    m = s["mask"].numpy()
    if m.ndim == 3:
        m = m[0]
    ax.imshow(img[0], cmap="inferno")
    if m.max() > 0:
        ax.contour((m > 0).astype(float), levels=[0.5], colors="lime", linewidths=0.9)
    crop = [int(x) for x in s["crop"].tolist()]
    ax.set_title(f"{s['video_id']}\ncrop={crop}", fontsize=8)
    ax.axis("off")
fig.suptitle(f"train=True | same index={idx} ×8 draws (augs + roi_random)", fontsize=12)
plt.tight_layout()
plt.show()
print("ok — если кропы/маски заметно разные, ауги и roi_random работают")


### Чеклист «датасет в порядке»

- [ ] у всех нужных `video_id` есть маска (`без маски` пусто или осознанно);
- [ ] `files_meta` / ROI есть там, где ожидаешь `roi_random`;
- [ ] бинарные маски: значения по сути 0/255 → в тензоре 0/1;
- [ ] на deep-dive красный `object_crop` покрывает пластину, жёлтый ROI — зону дефектов;
- [ ] контур маски на фичах совпадает с дефектами на сырых кадрах;
- [ ] `samples_per_video` и `len(ds)` сходятся; yaml тот, который правишь (`--yaml` / `YAML = ...`).
